# Battle-Smith RPG — Generator Gambar Otomatis di Google Colab (Gratis)

Selamat datang! Notebook ini akan membuatkan gambar untuk game Anda secara **otomatis dan gratis** memakai GPU Google Colab, lalu menyimpannya langsung ke **Google Drive** Anda.

### Paket yang Bisa Di-generate:
- `monsters` → **544 monster unik** (meng-cover semua 4.864 musuh di Excel)
- `materials` → **208 ikon material crafting**
- `trophies` → **480 ikon trofi monster**

--- 
### Panduan Singkat (Untuk Pemula):
1. **WAJIB nyalakan GPU dulu:** Klik menu atas **Runtime** → **Change runtime type** → pilih **T4 GPU** → klik **Save**.
2. **Jalankan kotak-kotak di bawah satu per satu dari nomor 1 sampai 6** dengan menekan tombol **▶ (Play)** di pojok kiri atas tiap kotak.
3. Di **Langkah 2**, pilih mau generate `monsters`, `materials`, atau `trophies` lalu klik **▶** (akan muncul pop-up izin Google Drive → klik **Connect to Google Drive** → **Allow**).
4. Di **Langkah 5**, awalnya sudah diset hanya area `Hollowfrost Range` dulu supaya cepat selesai (~1–2 menit) untuk tes. Kalau hasilnya sudah bagus, **hapus teks `Hollowfrost Range` hingga kosong** lalu klik **▶** lagi untuk men-generate **SEMUA area**!
5. **Tenang kalau terputus di tengah jalan:** Semua gambar langsung tersimpan ke Google Drive. Kalau berhenti, cukup jalankan ulang dari Langkah 1–5 — gambar yang sudah jadi **otomatis dilewati** (tidak mengulang dari awal).

In [ ]:
# @title Langkah 1 — Cek GPU & Siapkan Program (Tunggu ~1 menit)
import subprocess, sys

gpu_check = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if gpu_check.returncode != 0:
    raise RuntimeError(
        'GPU BELUM AKTIF! Silakan klik menu atas: Runtime -> Change runtime type -> pilih T4 GPU -> Save, lalu jalankan Langkah 1 lagi.'
    )
print('GPU aktif:', gpu_check.stdout.strip())
print('Sedang menginstall library AI (hanya perlu sekali per sesi)...')
!pip install -q --upgrade diffusers transformers accelerate safetensors pillow tqdm
print('Langkah 1 SELESAI! Lanjut ke Langkah 2.')

In [ ]:
# @title Langkah 2 — Pilih Jenis Gambar & Hubungkan ke Google Drive
from pathlib import Path
from google.colab import drive

# Pilih paket yang mau dibuat gambarnya:
ASSET_PACK = 'monsters'  # @param ['monsters', 'materials', 'trophies']

drive.mount('/content/drive')

DRIVE_BASE = Path('/content/drive/MyDrive/BattleSmith_GameArt')
OUT_DIR = DRIVE_BASE / ASSET_PACK
MANIFEST_PATH = DRIVE_BASE / f'{ASSET_PACK}_visual_manifest.json'

OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Google Drive terhubung!')
print('Paket dipilih :', ASSET_PACK)
print('Folder hasil  :', OUT_DIR)
print('Langkah 2 SELESAI! Lanjut ke Langkah 3.')

In [ ]:
# @title Langkah 3 — Unduh Daftar Prompt Otomatis dari GitHub Repo Anda
import json
import urllib.request
from google.colab import files

REFRESH_FROM_GITHUB = False  # @param {type:'boolean'}
GITHUB_RAW_URL = f'https://raw.githubusercontent.com/Gunawan5500/gameindes/main/data/{ASSET_PACK}_visual_manifest.json'

if REFRESH_FROM_GITHUB or not MANIFEST_PATH.exists():
    print('Mengunduh daftar prompt terbaru dari GitHub:', GITHUB_RAW_URL)
    try:
        urllib.request.urlretrieve(GITHUB_RAW_URL, MANIFEST_PATH)
        print('Berhasil diunduh & disimpan ke Google Drive:', MANIFEST_PATH)
    except Exception as e:
        print('Gagal unduh otomatis dari GitHub:', e)
        print(f'Silakan upload manual file {MANIFEST_PATH.name} dari komputer Anda:')
        uploaded = files.upload()
        first_name = next(iter(uploaded.keys()))
        MANIFEST_PATH.write_bytes(uploaded[first_name])

manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
entries = manifest['entries']
existing_count = sum(1 for e in entries if (OUT_DIR / e['filename']).exists())

print(f"Paket       : {ASSET_PACK} (Gaya: {manifest['style']})")
print(f"Total target: {len(entries)} gambar | Sudah jadi di Drive: {existing_count} | Sisa: {len(entries) - existing_count}")
print('Daftar Area :', sorted(set(e['area'] for e in entries)))
print('Langkah 3 SELESAI! Lanjut ke Langkah 4.')

In [ ]:
# @title Langkah 4 — Muat Model AI ke GPU (Mode Hemat VRAM untuk T4)
import gc
import torch
from diffusers import AutoPipelineForText2Image

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU tidak terdeteksi. Pastikan Runtime -> Change runtime type -> T4 GPU sudah aktif.')

# Bersihkan memori GPU dulu (aman jika Langkah 4 dijalankan lebih dari sekali)
if 'pipe' in globals():
    del pipe
gc.collect()
torch.cuda.empty_cache()

MODEL_ID = 'stabilityai/sdxl-turbo'
print('Sedang memuat model AI (SDXL-Turbo) dalam mode hemat VRAM...')

pipe = AutoPipelineForText2Image.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    variant='fp16',
    use_safetensors=True,
    low_cpu_mem_usage=True,
)
# CPU offload otomatis memindahkan komponen ke GPU hanya saat dibutuhkan (VRAM jauh lebih hemat & anti-OOM di T4!)
pipe.enable_model_cpu_offload()
pipe.enable_attention_slicing()
pipe.enable_vae_slicing()
pipe.set_progress_bar_config(disable=True)
print('Model AI SIAP (mode hemat VRAM)! Lanjut ke Langkah 5.')

In [ ]:
# @title Langkah 5 — Mulai Generate Gambar! (Otomatis Simpan ke Google Drive)
from IPython.display import display
from tqdm.auto import tqdm

# --- PENGATURAN MUDAH ---
# Untuk uji coba awal, biarkan 'Hollowfrost Range' (~1-2 menit).
# Jika hasil sudah cocok, HAPUS teksnya hingga kosong ('') untuk generate SEMUA 16 Area!
AREA_FILTER = 'Hollowfrost Range'  # @param {type:'string'}
BOSS_ONLY = False  # @param {type:'boolean'}
LIMIT = 0  # @param {type:'integer'}
NUM_STEPS = 4  # @param {type:'slider', min:2, max:8, step:1}
# 'auto' otomatis menyimpan 512x512 untuk monsters & 256x256 untuk materials/trophies:
SAVE_SIZE = 'auto'  # @param ['auto', '256', '512']
WEBP_QUALITY = 85  # @param {type:'slider', min:60, max:95, step:5}
SHOW_PREVIEW_EVERY = 10  # @param {type:'integer'}
OVERWRITE = False  # @param {type:'boolean'}

target_px = (512 if ASSET_PACK == 'monsters' else 256) if SAVE_SIZE == 'auto' else int(SAVE_SIZE)

selected = entries
if AREA_FILTER.strip():
    selected = [e for e in selected if e['area'].lower() == AREA_FILTER.strip().lower()]
if BOSS_ONLY:
    selected = [e for e in selected if e['tipe'] in ('Boss', 'Mini Boss')]

remaining = [e for e in selected if OVERWRITE or not (OUT_DIR / e['filename']).exists()]
print(f'Target terpilih: {len(selected)} | Sudah ada (dilewati): {len(selected) - len(remaining)} | Akan dibuat sekarang: {len(remaining)}')
print(f'Ukuran simpan  : {target_px}x{target_px} px (.webp)')

generated = 0
skipped = 0

for entry in tqdm(selected, desc=f'Generating {ASSET_PACK}'):
    dest = OUT_DIR / entry['filename']
    if dest.exists() and not OVERWRITE:
        skipped += 1
        continue
    if LIMIT > 0 and generated >= LIMIT:
        break

    dest.parent.mkdir(parents=True, exist_ok=True)
    gen = torch.Generator(device='cpu').manual_seed(int(entry['seed']))

    with torch.inference_mode():
        image = pipe(
            prompt=entry['prompt'],
            num_inference_steps=NUM_STEPS,
            guidance_scale=0.0,
            width=512,
            height=512,
            generator=gen,
        ).images[0]

    out_img = image.resize((target_px, target_px)) if target_px != 512 else image
    out_img.save(dest, format='WEBP', quality=WEBP_QUALITY, method=6)
    generated += 1

    if SHOW_PREVIEW_EVERY > 0 and (generated == 1 or generated % SHOW_PREVIEW_EVERY == 0):
        print(f"\nPreview #{generated}: {entry['display_name']} ({entry['area']} - {entry['tipe']})")
        display(image.resize((220, 220)))

print(f'\nLangkah 5 SELESAI! Baru dibuat: {generated} | Dilewati (sudah ada): {skipped}')
print('Semua gambar tersimpan aman di Google Drive Anda:', OUT_DIR)

In [ ]:
# @title Langkah 6 — Bungkus Semua Gambar Jadi 1 File ZIP (Siap Download)
import shutil

zip_base = DRIVE_BASE / f'battle_smith_{ASSET_PACK}_art'
zip_file = shutil.make_archive(str(zip_base), 'zip', root_dir=OUT_DIR)
count = sum(1 for _ in OUT_DIR.rglob('*.webp'))
print(f'Total file .webp di dalam ZIP ({ASSET_PACK}): {count}')
print(f'File ZIP siap diunduh dari Google Drive Anda: {zip_file}')
print('Buka Google Drive -> folder BattleSmith_GameArt -> klik kanan file .zip -> Download!')